## Partie 1 : Exploration du dataset
On écrit une fonction `get_image_info` qui extrait, pour chaque image : nom, classe, format, mode, largeur, hauteur, écart-type des pixels, nombre de canaux et taille du fichier.
Les fichiers corrompus sont pris en charge avec un `try/except` : on garde leur ligne avec `corrupted=True` et les autres valeurs vides.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

In [5]:
def get_image_info(path: Path, label: str) -> dict:
    # Dictionnaire qui va contenir toutes les informations de l'image
    info = {
        # Nom du fichier
        "name": path.name,

        # Classe de l'image (ex: glass, plastic, metal...)
        "class": label,

        # Chemin complet de l'image
        "path": str(path),

        # Informations qui seront récupérées après ouverture de l'image
        "format": None,
        "mode": None,
        "width": None,
        "height": None,
        "std": None,
        "channels": None,

        # Taille du fichier en Ko
        "size_kb": round(path.stat().st_size / 1024, 2),

        # On considère l'image comme valide au départ
        "corrupted": False,
    }

    try:
        # Ouvre l'image avec Pillow
        with Image.open(path) as img:

            # Force le chargement complet de l'image
            # Cela permet notamment de détecter les fichiers tronqués ou corrompus
            img.load()

            # Convertit l'image en tableau NumPy
            arr = np.array(img)

            # Récupération des caractéristiques de l'image
            info.update({
                # Format du fichier : JPEG, PNG, WEBP...
                "format": img.format,

                # Mode de l'image : RGB, L, RGBA...
                "mode": img.mode,

                # Largeur de l'image en pixels
                "width": img.width,

                # Hauteur de l'image en pixels
                "height": img.height,

                # Écart-type des valeurs des pixels
                "std": float(arr.std()),

                # Nombre de canaux :
                # 1 si l'image est en niveaux de gris (2 dimensions)
                # sinon on récupère le nombre de canaux dans la 3e dimension
                "channels": 1 if arr.ndim == 2 else arr.shape[2],
            })

    # Si une erreur se produit lors de l'ouverture ou du chargement
    # l'image est considérée comme corrompue
    except Exception:
        info["corrupted"] = True

    # Retourne toutes les informations de l'image
    return info

On définit le chemin du dossier `raw` et la liste des classes, puis on parcourt chaque sous-dossier pour appliquer `get_image_info` à chaque fichier. Le résultat est rassemblé dans un DataFrame `df`.

In [6]:
RAW_DIR = Path("../data/raw")
CLASSES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]

rows = []
for label in CLASSES:
    for path in sorted((RAW_DIR / label).glob("*")):
        if path.is_file():
            rows.append(get_image_info(path, label))

df = pd.DataFrame(rows)
print(df.shape)
df.head()

(1032, 11)


,name,class,path,format,mode,width,height,std,channels,size_kb,corrupted
0,cardboard1.jpg,cardboard,..\data\raw\cardboard\cardboard1.jpg,JPEG,RGB,512.0,384.0,40.588529,3.0,16.93,False
1,cardboard10.jpg,cardboard,..\data\raw\cardboard\cardboard10.jpg,JPEG,RGB,512.0,384.0,42.571288,3.0,21.17,False
2,cardboard100.jpg,cardboard,..\data\raw\cardboard\cardboard100.jpg,JPEG,RGB,512.0,384.0,46.108305,3.0,14.54,False
3,cardboard101.jpg,cardboard,..\data\raw\cardboard\cardboard101.jpg,JPEG,RGB,512.0,384.0,72.263996,3.0,13.95,False
4,cardboard102.jpg,cardboard,..\data\raw\cardboard\cardboard102.jpg,JPEG,RGB,512.0,384.0,48.388937,3.0,17.59,False
